# Recipe 8: Transformers and Large Language Models
**EDPY 506 · Machine Learning: Theory and Applications · Recipe notebooks**

<a href="https://colab.research.google.com/github/okanbulut/edpy506/blob/main/recipes/notebooks/Recipe08_Transformers_LLMs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

A **transformer** reads every word in light of every other word (*attention*), so "bank" means something different in "river bank" and "bank loan". Models pretrained on huge amounts of text can be used three ways: (1) as an **embedding** machine whose numbers feed a simple classifier, (2) **fine-tuned** on labelled examples, or (3) **prompted** with instructions and a few examples. This recipe compares route 1 and route 3 with the TF-IDF model from Recipe 7.

**Lectures:** 10 (Transformers and LLMs)
**Data:** SMS Spam Collection (Almeida & Hidalgo, 2011): 5,572 real text messages labelled spam or ham (not spam), the same data as the Lecture 9 hands-on. It loads straight from GitHub. The recipe is identical for educational text such as forum posts, open-ended answers, or course reviews.
**Runtime:** about 5 minutes with a GPU, 10 to 15 minutes on a CPU. No setup or uploads are needed: just run the cells from top to bottom (**Runtime → Run all**).

**Faster with a GPU:** Runtime → Change runtime type → **T4 GPU** (free). Models download from Hugging Face the first time; no account or API key is needed.

**How this notebook is organized.** It follows the seven steps of the *ML Workflow Recipe* paper:
1. Define the problem · 2. Split the data · 3. Preprocess · 4. Baseline · 5. Train and tune · 6. Evaluate · 7. Interpret

After each output there is a **What you should see** note. Your numbers should match closely if you keep the default parameters. If you change the parameters, the numbers will change, and that is the point: try it and see what happens.

In [ ]:
# Run this cell first. It loads the tools (transformers is already installed in Colab).
import time, re, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU (slower, but it works)")

## ✏️ Change these parameters
This is the **only cell you need to edit**. Change a value, then choose **Runtime → Run all** to rerun the whole recipe.

In [ ]:
# ===== CHANGE THESE PARAMETERS =====
SEED = 506
SAMPLE_SIZE = 2000                    # messages to use (all 5,572 takes longer; 1000 is quicker)
TEST_SIZE = 0.3                       # share kept aside for the test
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"   # small, fast sentence-embedding model
CHAT_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"                # small chat model; "Qwen/Qwen2.5-1.5B-Instruct" is better but slower
N_PROMPTED = 40                       # how many test messages the chat model labels (it is slow)
SEARCH_QUERY = "you have won money"   # semantic search: find messages that MEAN this
# ====================================

## Load the data

In [ ]:
URL = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
sms = pd.read_csv(URL, sep="\t", header=None, names=["label", "text"])
sms["spam"] = (sms["label"] == "spam").astype(int)
sms = sms.sample(min(SAMPLE_SIZE, len(sms)), random_state=SEED).reset_index(drop=True)
print(f"Using {len(sms):,} messages; {sms['spam'].mean():.1%} spam.")

## Step 1. Define the problem
Same question as Recipe 7: spam or not? Keeping the question fixed lets us compare **representations** (counts vs. transformer embeddings) and **routes** (training a classifier vs. prompting).

## Step 2. Split the data

In [ ]:
from sklearn.model_selection import train_test_split
train, test = train_test_split(sms, test_size=TEST_SIZE, random_state=SEED, stratify=sms["spam"])
print(f"Training: {len(train):,}   Test: {len(test):,}")

## Steps 3 and 4. Baseline: TF-IDF + logistic regression (Recipe 7)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, precision_score

results = {}
def score(name, y_true, y_pred, seconds):
    results[name] = {"accuracy": accuracy_score(y_true, y_pred), "spam recall": recall_score(y_true, y_pred),
                     "spam precision": precision_score(y_true, y_pred, zero_division=0), "seconds": seconds}

t0 = time.time()
tfidf = TfidfVectorizer(min_df=2)
tfidf_clf = LogisticRegression(max_iter=2000, C=10).fit(tfidf.fit_transform(train["text"]), train["spam"])
score("TF-IDF + logistic", test["spam"], tfidf_clf.predict(tfidf.transform(test["text"])), time.time() - t0)
pd.DataFrame(results).T.round(3)

> **What you should see:** 2,000 messages (12.8% spam): 1,400 for training and 600 for testing. TF-IDF + logistic regression gets about 97.7% accuracy, spam recall of about 0.86 and precision of about 0.96, in a few hundredths of a second.

## Step 5a. Route 1: transformer embeddings + logistic regression
The transformer turns each message into **384 numbers** that capture its meaning. We average the numbers for all the tokens (*mean pooling*), then train the same logistic regression on them.

In [ ]:
from transformers import AutoTokenizer, AutoModel
tok = AutoTokenizer.from_pretrained(EMBED_MODEL)
enc_model = AutoModel.from_pretrained(EMBED_MODEL).to(DEVICE).eval()

def embed(texts, batch_size=64):
    """Turn a list of texts into one vector each (mean of the token vectors), scaled to length 1."""
    out = []
    for i in range(0, len(texts), batch_size):
        batch = tok(list(texts[i:i + batch_size]), padding=True, truncation=True, max_length=128, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            hidden = enc_model(**batch).last_hidden_state
        mask = batch["attention_mask"].unsqueeze(-1)
        vecs = (hidden * mask).sum(1) / mask.sum(1)
        out.append(torch.nn.functional.normalize(vecs, dim=1).cpu().numpy())
    return np.vstack(out)

t0 = time.time()
E_train, E_test = embed(train["text"].tolist()), embed(test["text"].tolist())
emb_clf = LogisticRegression(max_iter=2000, C=10).fit(E_train, train["spam"])
score("Transformer embeddings + logistic", test["spam"], emb_clf.predict(E_test), time.time() - t0)
print(f"Each message is now {E_train.shape[1]} numbers.")
pd.DataFrame(results).T.round(3)

> **What you should see:** Each message becomes 384 numbers. Embeddings + logistic regression edge ahead: about 98.3% accuracy, spam recall of about 0.88 and precision of about 0.99. Embedding the messages takes about 5 seconds on a laptop CPU.

### Embeddings capture meaning: semantic search
Type a query in `SEARCH_QUERY`. We embed it and find the messages whose vectors point in the most similar direction, even if they share **no words** with the query.

In [ ]:
q = embed([SEARCH_QUERY])[0]
sims = E_train @ q
hits = train.assign(similarity=sims).sort_values("similarity", ascending=False).head(5)
hits[["similarity", "label", "text"]].round(2)

> **What you should see:** All five hits are spam, with similarities from about 0.71 down to 0.55. The top hit is "Money i have won wining number 946 wot do i do next". Read the other hits: how many of them share words with your query?

## Step 5b. Route 3: prompting a small chat model
No training at all. We give the model an instruction and two examples (*few-shot prompting*) and read its answer. Because generating text is slow, it labels only `N_PROMPTED` test messages.

In [ ]:
from transformers import pipeline
chat = pipeline("text-generation", model=CHAT_MODEL, device_map="auto", dtype="auto")

SYSTEM = "You are a spam filter for text messages. Answer with exactly one word: spam or ham."
EXAMPLES = [("WINNER!! You have been selected to receive a £900 prize. Call 09061701461 to claim.", "spam"),
            ("Ok, I'll pick you up at 6. Don't forget the book.", "ham")]

def ask(message):
    msgs = [{"role": "system", "content": SYSTEM}]
    for text, label in EXAMPLES:
        msgs += [{"role": "user", "content": text}, {"role": "assistant", "content": label}]
    msgs.append({"role": "user", "content": message})
    reply = chat(msgs, max_new_tokens=4, do_sample=False)[0]["generated_text"][-1]["content"]
    return 1 if "spam" in reply.lower() else 0

few = test.groupby("spam", group_keys=False).head(N_PROMPTED // 2)   # half spam, half ham
t0 = time.time()
few = few.assign(prompted=[ask(m) for m in few["text"]])
secs = time.time() - t0
score(f"Prompted chat model ({len(few)} msgs)", few["spam"], few["prompted"], secs)
# For a fair comparison, score the other two models on the same messages
idx = [test.index.get_loc(i) for i in few.index]
score(f"TF-IDF on same {len(few)}", few["spam"], tfidf_clf.predict(tfidf.transform(few["text"])), np.nan)
score(f"Embeddings on same {len(few)}", few["spam"], emb_clf.predict(E_test[idx]), np.nan)
print(f"The chat model took {secs / len(few):.2f} seconds per message.")
pd.DataFrame(results).T.round(3)

> **What you should see:** A surprise worth discussing: the small 0.5B chat model gets only about **43%** of the balanced 40 messages right, worse than a coin flip. It catches most spam (recall 0.80) but calls many ordinary messages spam too (precision about 0.46). On the same 40 messages, TF-IDF gets about 93% and the embeddings about 95%. The chat model took about 0.04 seconds per message on a laptop with a GPU; on a CPU expect a few tenths of a second or more.

## Step 6. Evaluate: where does the chat model go wrong?

In [ ]:
wrong = few[few["prompted"] != few["spam"]]
print(f"{len(wrong)} of {len(few)} wrong.")
wrong[["label", "text"]].head(8)

> **What you should see:** About 23 of the 40 are wrong, and most mistakes (about 19) are ordinary messages labelled spam, such as "It could work, we'll reach a consensus at the next meeting" and "Nope, I'm still in the market". A tiny general-purpose model with two examples is not a reliable filter; a bigger model (try the 1.5B one) or a trained classifier does much better.

## Step 7. Interpret: choosing a route
| Route | Needs labelled data? | Speed | Explainable? | Best when |
|---|---|---|---|---|
| TF-IDF + logistic | Yes, hundreds+ | Very fast | Yes: word weights | Plenty of labels; wording matters |
| Embeddings + logistic | Yes, but fewer | Fast after one pass | Partly | Meaning matters more than exact words |
| Prompting an LLM | No (a few examples) | Slow, costly at scale | It can give reasons, but they may not be faithful | Few labels; open-ended tasks; quick prototypes |

Always check an LLM's labels against human labels on a sample before trusting it (Lecture 10), and audit any model that affects people (Recipe 9).

## Try this next
1. Change `CHAT_MODEL` to `"Qwen/Qwen2.5-1.5B-Instruct"` (use a GPU). Does a bigger model label better?
2. Delete the two `EXAMPLES` (make the list empty) to go **zero-shot**. Does accuracy drop?
3. Change `SEARCH_QUERY` to `"are you coming to dinner"`. Do the hits share words with your query?
4. Set `SAMPLE_SIZE = 500`. With fewer training messages, which route holds up best?

---
## Where to go next
- **Companion paper:** *Companion papers/Recipe08_Transformers_LLMs_Companion.pdf* explains the idea behind this recipe in plain words.
- **Workflow:** *01_ML_Workflow_Recipe.pdf* has a checklist for every step used here.
- **Which method?** *02_Which_Method_Flowchart.pdf* helps you pick the right recipe for a new question.
- **Lecture 10 hands-on notebooks** compare a fine-tuned BERT, zero-shot labels, and prompting on Coursera reviews, and summarize news articles.

*Data citation:* Almeida, T. A., Gómez Hidalgo, J. M., & Yamakami, A. (2011). Contributions to the study of SMS spam filtering. *Proceedings of the 11th ACM Symposium on Document Engineering*, 259–262.